In [5]:
import pandas as pd
import numpy as np
import sys

# Load raw dataset
RAW_PATH = "/content/drive/MyDrive/Data Science Project Lifecycle/Individual Coursework/Agricultural land (% of land area).csv"

print("=" * 65)
print("    Requirements Gathering & Dataset Confirmation")
print("=" * 65)

df_raw = pd.read_csv(RAW_PATH)
year_cols = [c for c in df_raw.columns if c.isdigit()]

# Melt to long format
df_long = df_raw[["REF_AREA_LABEL", "REF_AREA"] + year_cols].melt(
    id_vars=["REF_AREA_LABEL", "REF_AREA"],
    var_name="Year",
    value_name="Agri_Land_Pct",
)
df_long.columns = ["Country", "Country_Code", "Year", "Agri_Land_Pct"]
df_long["Year"] = df_long["Year"].astype(int)
df_long = df_long.dropna(subset=["Agri_Land_Pct"]).reset_index(drop=True)

# Remove regional/aggregate rows (not individual countries)
AGGREGATE_PATTERNS = [
    "income", "IDA", "IBRD", "OECD", "World", "Sub-Saharan",
    "East Asia", "South Asia", "North America", "Latin America",
    "Caribbean", "Europe &", "Middle East, North Africa",
    "Fragile", "small states", "Pacific island", "Arab World",
]
mask_agg = df_long["Country"].str.contains(
    "|".join(AGGREGATE_PATTERNS), case=False, na=False
)
df_countries = df_long[~mask_agg].copy()

# CW requirement checks
print("\n COURSEWORK REQUIREMENTS CHECKLIST")
print("-" * 65)

total_obs = len(df_countries)
req_obs_ok = total_obs >= 100
print(f"  Min 100 observations:  {total_obs:,} rows  {'PASS' if req_obs_ok else 'FAIL'}")

unique_countries = df_countries["Country"].nunique()
unique_years     = df_countries["Year"].nunique()
dims             = 4  # Country, Country_Code, Year, Agri_Land_Pct
req_dims_ok = dims >= 3
print(f"  Min 3 dimensions:      {dims} columns (Country, Code, Year, Value)  {'PASS' if req_dims_ok else 'FAIL'}")

print(f"  Source:                World Bank — World Development Indicators (WDI)")
print(f"  Sustainability topic:  Agricultural land use — directly sustainability-relevant")

print(f"\n DATASET SUMMARY")
print("-" * 65)
print(f"  Countries / territories : {unique_countries}")
print(f"  Years covered           : {df_countries['Year'].min()} – {df_countries['Year'].max()}  ({unique_years} years)")
print(f"  Total observations      : {total_obs:,}")
print(f"  Value range             : {df_countries['Agri_Land_Pct'].min():.2f}% – {df_countries['Agri_Land_Pct'].max():.2f}%")
print(f"  Global mean (all years) : {df_countries['Agri_Land_Pct'].mean():.2f}%")

print("\n FUNCTIONAL REQUIREMENTS (FR)")
print("-" * 65)
FRs = [
    "FR1  The dashboard shall display a choropleth world map of agricultural land %",
    "FR2  The dashboard shall allow filtering by year range using a slider",
    "FR3  The dashboard shall allow filtering by continent/region using a multiselect",
    "FR4  The dashboard shall display a global trend line chart (average over time)",
    "FR5  The dashboard shall display a country-level deep-dive trend chart",
    "FR6  The dashboard shall display a top-N / bottom-N country bar chart",
    "FR7  The dashboard shall display a regional box-plot distribution chart",
    "FR8  The dashboard shall display KPI summary cards at the top of the page",
]
for fr in FRs:
    print(f"  {fr}")

print("\n NON-FUNCTIONAL REQUIREMENTS (NFR)")
print("-" * 65)
NFRs = [
    "NFR1 The app shall load within 5 seconds on a standard connection",
    "NFR2 The app shall be publicly accessible via a Streamlit Cloud URL",
    "NFR3 The dashboard shall be compatible with Chrome and Firefox",
    "NFR4 The app source code shall be version-controlled in a public GitHub repo",
    "NFR5 The app shall display no unhandled Python exceptions during normal use",
]
for nfr in NFRs:
    print(f"  {nfr}")

print("\n  Dataset confirmed — proceed to Step 2: Data Acquisition & Cleaning\n")


    Requirements Gathering & Dataset Confirmation

 COURSEWORK REQUIREMENTS CHECKLIST
-----------------------------------------------------------------
  Min 100 observations:  12,740 rows  PASS
  Min 3 dimensions:      4 columns (Country, Code, Year, Value)  PASS
  Source:                World Bank — World Development Indicators (WDI)
  Sustainability topic:  Agricultural land use — directly sustainability-relevant

 DATASET SUMMARY
-----------------------------------------------------------------
  Countries / territories : 220
  Years covered           : 1961 – 2023  (63 years)
  Total observations      : 12,740
  Value range             : 0.26% – 93.44%
  Global mean (all years) : 37.07%

 FUNCTIONAL REQUIREMENTS (FR)
-----------------------------------------------------------------
  FR1  The dashboard shall display a choropleth world map of agricultural land %
  FR2  The dashboard shall allow filtering by year range using a slider
  FR3  The dashboard shall allow filtering by con

**Data retrieval & Cleaning**

In [8]:
import pandas as pd
import numpy as np
import os

RAW_PATH   = "/content/drive/MyDrive/Data Science Project Lifecycle/Individual Coursework/Agricultural land (% of land area).csv"
CLEAN_PATH = "/content/drive/MyDrive/Data Science Project Lifecycle/Individual Coursework/Agricultural land_clean.csv"

print("=" * 65)
print("   Data retrieval & Cleaning")
print("=" * 65)

# Load raw data
print("\n[1] Loading raw dataset")
df_raw = pd.read_csv(RAW_PATH)
year_cols = [c for c in df_raw.columns if c.isdigit()]

print(f"  Raw shape            : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"  Indicator            : {df_raw['INDICATOR_LABEL'].iloc[0]}")
print(f"  Unit                 : {df_raw['UNIT_MEASURE_LABEL'].iloc[0]}")
print(f"  Year columns found   : {year_cols[0]} – {year_cols[-1]}  ({len(year_cols)} years)")

# Select relevant columns & reshape to long format
print("\n[2] Reshaping from wide to long format …")
df_long = df_raw[["REF_AREA_LABEL", "REF_AREA"] + year_cols].melt(
    id_vars=["REF_AREA_LABEL", "REF_AREA"],
    var_name="Year",
    value_name="Agri_Land_Pct",
)
df_long.columns = ["Country", "Country_Code", "Year", "Agri_Land_Pct"]
df_long["Year"] = df_long["Year"].astype(int)
print(f"  Long-format shape    : {df_long.shape[0]:,} rows × {df_long.shape[1]} columns")

# Remove aggregate / regional rows (keep individual countries)
print("\n[3] Removing regional aggregates (not individual countries) …")
AGGREGATE_PATTERNS = [
    "income", "IDA", "IBRD", "OECD", "World", "Sub-Saharan",
    "East Asia", "South Asia", "North America", "Latin America",
    "Caribbean", "Europe &", "Middle East, North Africa",
    "Fragile", "small states", "Pacific island", "Arab World",
]
mask_agg = df_long["Country"].str.contains(
    "|".join(AGGREGATE_PATTERNS), case=False, na=False
)
removed = mask_agg.sum()
df_countries = df_long[~mask_agg].copy()
print(f"  Aggregate rows removed : {removed}")
print(f"  Individual-country rows: {len(df_countries):,}")
print(f"  Unique countries       : {df_countries['Country'].nunique()}")

# Handle missing values
print("\n[4] Handling missing values …")
total_cells  = len(df_countries)
missing_before = df_countries["Agri_Land_Pct"].isna().sum()
missing_pct    = missing_before / total_cells * 100
print(f"  Missing before  : {missing_before:,}  ({missing_pct:.1f}%)")
print(f"  Strategy        : Drop rows with missing Agri_Land_Pct")
print(f"  (Missing values are mainly early years for post-independence countries)")
df_clean = df_countries.dropna(subset=["Agri_Land_Pct"]).reset_index(drop=True)
missing_after = df_clean["Agri_Land_Pct"].isna().sum()
print(f"  Missing after   : {missing_after}")
print(f"  Rows retained   : {len(df_clean):,}")

# Add continent/region column (manual mapping using Country_Code)
print("\n[5] Adding Region column …")

REGION_MAP = {
    # Africa
    "DZA":"Africa","AGO":"Africa","BEN":"Africa","BWA":"Africa","BFA":"Africa",
    "BDI":"Africa","CPV":"Africa","CMR":"Africa","CAF":"Africa","TCD":"Africa",
    "COM":"Africa","COD":"Africa","COG":"Africa","CIV":"Africa","DJI":"Africa",
    "EGY":"Africa","GNQ":"Africa","ERI":"Africa","SWZ":"Africa","ETH":"Africa",
    "GAB":"Africa","GMB":"Africa","GHA":"Africa","GIN":"Africa","GNB":"Africa",
    "KEN":"Africa","LSO":"Africa","LBR":"Africa","LBY":"Africa","MDG":"Africa",
    "MWI":"Africa","MLI":"Africa","MRT":"Africa","MUS":"Africa","MAR":"Africa",
    "MOZ":"Africa","NAM":"Africa","NER":"Africa","NGA":"Africa","RWA":"Africa",
    "STP":"Africa","SEN":"Africa","SLE":"Africa","SOM":"Africa","ZAF":"Africa",
    "SSD":"Africa","SDN":"Africa","TZA":"Africa","TGO":"Africa","TUN":"Africa",
    "UGA":"Africa","ZMB":"Africa","ZWE":"Africa","MYT":"Africa","REU":"Africa",
    "SHN":"Africa","ESH":"Africa",
    # Asia
    "AFG":"Asia","ARM":"Asia","AZE":"Asia","BHR":"Asia","BGD":"Asia","BTN":"Asia",
    "BRN":"Asia","KHM":"Asia","CHN":"Asia","GEO":"Asia","IND":"Asia","IDN":"Asia",
    "IRN":"Asia","IRQ":"Asia","ISR":"Asia","JPN":"Asia","JOR":"Asia","KAZ":"Asia",
    "KWT":"Asia","KGZ":"Asia","LAO":"Asia","LBN":"Asia","MYS":"Asia","MDV":"Asia",
    "MNG":"Asia","MMR":"Asia","NPL":"Asia","PRK":"Asia","OMN":"Asia","PAK":"Asia",
    "PSE":"Asia","PHL":"Asia","QAT":"Asia","SAU":"Asia","SGP":"Asia","KOR":"Asia",
    "LKA":"Asia","SYR":"Asia","TWN":"Asia","TJK":"Asia","THA":"Asia","TLS":"Asia",
    "TUR":"Asia","TKM":"Asia","ARE":"Asia","UZB":"Asia","VNM":"Asia","YEM":"Asia",
    # Europe
    "ALB":"Europe","AND":"Europe","AUT":"Europe","BLR":"Europe","BEL":"Europe",
    "BIH":"Europe","BGR":"Europe","HRV":"Europe","CYP":"Europe","CZE":"Europe",
    "DNK":"Europe","EST":"Europe","FIN":"Europe","FRA":"Europe","DEU":"Europe",
    "GRC":"Europe","HUN":"Europe","ISL":"Europe","IRL":"Europe","ITA":"Europe",
    "XKX":"Europe","LVA":"Europe","LIE":"Europe","LTU":"Europe","LUX":"Europe",
    "MLT":"Europe","MDA":"Europe","MCO":"Europe","MNE":"Europe","NLD":"Europe",
    "MKD":"Europe","NOR":"Europe","POL":"Europe","PRT":"Europe","ROU":"Europe",
    "RUS":"Europe","SMR":"Europe","SRB":"Europe","SVK":"Europe","SVN":"Europe",
    "ESP":"Europe","SWE":"Europe","CHE":"Europe","UKR":"Europe","GBR":"Europe",
    "VAT":"Europe","FRO":"Europe","GIB":"Europe","GGY":"Europe","IMN":"Europe",
    "JEY":"Europe","KOS":"Europe",
    # North America
    "ATG":"North America","BHS":"North America","BRB":"North America",
    "BLZ":"North America","CAN":"North America","CRI":"North America",
    "CUB":"North America","DMA":"North America","DOM":"North America",
    "SLV":"North America","GRD":"North America","GTM":"North America",
    "HTI":"North America","HND":"North America","JAM":"North America",
    "MEX":"North America","NIC":"North America","PAN":"North America",
    "KNA":"North America","LCA":"North America","VCT":"North America",
    "TTO":"North America","USA":"North America","BMU":"North America",
    "CYM":"North America","GRL":"North America","MTQ":"North America",
    "PRI":"North America","TCA":"North America","VIR":"North America",
    # South America
    "ARG":"South America","BOL":"South America","BRA":"South America",
    "CHL":"South America","COL":"South America","ECU":"South America",
    "GUY":"South America","PRY":"South America","PER":"South America",
    "SUR":"South America","URY":"South America","VEN":"South America",
    "GUF":"South America",
    # Oceania
    "AUS":"Oceania","FJI":"Oceania","KIR":"Oceania","MHL":"Oceania",
    "FSM":"Oceania","NRU":"Oceania","NZL":"Oceania","PLW":"Oceania",
    "PNG":"Oceania","WSM":"Oceania","SLB":"Oceania","TON":"Oceania",
    "TUV":"Oceania","VUT":"Oceania","NCL":"Oceania","PYF":"Oceania",
    "GUM":"Oceania","MNP":"Oceania","ASM":"Oceania","COK":"Oceania",
    "NIU":"Oceania","TKL":"Oceania","WLF":"Oceania",
}

df_clean["Region"] = df_clean["Country_Code"].map(REGION_MAP).fillna("Other")
region_counts = df_clean[df_clean["Year"] == 2022]["Region"].value_counts()
print(f"  Regions assigned:")
for region, count in region_counts.items():
    print(f"    {region:<18}: {count} countries")

# Add derived columns
print("\n[6] Adding derived columns …")
df_clean = df_clean.sort_values(["Country", "Year"]).reset_index(drop=True)
df_clean["YoY_Change"] = df_clean.groupby("Country")["Agri_Land_Pct"].diff()
print("  Added: YoY_Change (year-on-year percentage-point change)")

# Final validation & save
print("\n[7] Final validation …")
assert df_clean["Agri_Land_Pct"].isna().sum() == 0, "Still has missing values!"
assert df_clean["Year"].between(1961, 2023).all(),  "Year out of expected range!"
assert df_clean["Agri_Land_Pct"].between(0, 100).all(), "Value out of 0-100 range!"

print(f"  No missing values in target column")
print(f"  All years in valid range")
print(f"  All values in 0–100% range")
print(f"\n  Final clean dataset shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"  Columns: {list(df_clean.columns)}")

df_clean.to_csv(CLEAN_PATH, index=False)

   Data retrieval & Cleaning

[1] Loading raw dataset
  Raw shape            : 258 rows × 102 columns
  Indicator            : Agricultural land (% of land area)
  Unit                 : Percentage of land area
  Year columns found   : 1961 – 2023  (63 years)

[2] Reshaping from wide to long format …
  Long-format shape    : 16,254 rows × 4 columns

[3] Removing regional aggregates (not individual countries) …
  Aggregate rows removed : 2394
  Individual-country rows: 13,860
  Unique countries       : 220

[4] Handling missing values …
  Missing before  : 1,120  (8.1%)
  Strategy        : Drop rows with missing Agri_Land_Pct
  (Missing values are mainly early years for post-independence countries)
  Missing after   : 0
  Rows retained   : 12,740

[5] Adding Region column …
  Regions assigned:
    Africa            : 53 countries
    Asia              : 47 countries
    Europe            : 45 countries
    North America     : 28 countries
    Oceania           : 19 countries
    Other  